In [141]:
import duckdb

import dash
import dash_core_components as dcc
import dash_html_components as html
import dash_table
import pandas as pd
import plotly.express as px
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import plotly.graph_objects as go


# ─── Load data ───


In [142]:
matches = '''
SELECT
  *,
  CASE
    WHEN STARTS_WITH(match_id, 'NA1_') THEN 'americas'
    ELSE 'europe'
  END AS region
FROM read_parquet('data/matches.parquet')
'''
matches = duckdb.query(matches).to_df()
matches['duration_min'] = round(matches['game_duration_sec'] / 60)
matches['win_num'] = matches['win'].astype(int)

players = '''
SELECT *
FROM read_parquet('data/players.parquet')
'''
players = duckdb.query(players).to_df()

items = '''
SELECT *
FROM read_parquet('data/items.parquet')
'''
items = duckdb.query(items).to_df()

item_cols = ['item0', 'item1', 'item2', 'item3', 'item4', 'item5']
item_dict = dict(zip(items['item_id'], items['item_name']))

for col in item_cols:
    matches[col] = matches[col].fillna(0).astype(int).astype(str)
    matches[col] = matches[col].map(item_dict)

In [143]:
items_full = pd.melt(matches, id_vars=['win', 'team_position'], 
                value_vars=[f'item{i}' for i in range(6)], 
                var_name='slot', value_name='item_id')
items_full = items_full[items_full['item_id'] != 0]  # убираем пустые слоты

win_items = items_full[items_full['win'] == True]['item_id']
top10 = win_items.value_counts().head(10).reset_index()
top10.columns = ['item_id', 'count']

matches['kda'] = round((matches['kills'].mean() + matches['assists'].mean()) / matches['deaths'].replace(0,1), 2)

role_counts = items_full[items_full['win'] == True].groupby(['team_position', 'item_id']).size().reset_index(name='count')
role_top5 = role_counts.sort_values(['team_position', 'count'], ascending=[True, False]) \
                       .groupby('team_position').head(5).reset_index(drop=True)
POSITIONS = matches['team_position'].dropna().unique().tolist()
POS_COLORS = {'TOP': '#ef4444', 'JUNGLE': '#22c55e', 'MIDDLE': '#3b82f6',
              'BOTTOM': '#f59e0b', 'UTILITY': '#a855f7'}

# ─── App ───

In [144]:
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.DARKLY]
)

# ─── Filters ───

In [145]:
region_selector = dcc.Dropdown(
    id='region_selector',
    options=[{'label': r, 'value': r} for r in matches['region'].unique()],
    value=['europe', 'americas'],
    multi=True,
    className='dash-dropdown'
)

position_selector = dcc.Dropdown(
    id='position_selector',
    options=[{'label': p, 'value': p} for p in POSITIONS],
    value=POSITIONS,
    multi=True,
    placeholder='Все роли',
    className='dash-dropdown'
)

win_selector = dcc.Dropdown(
    id='win_selector',
    options=[{'label': 'Победа', 'value': 1},
             {'label': 'Поражение', 'value': 0},
             {'label': 'Все', 'value': 'all'}],
    value='all',
    clearable=False,
    className='dash-dropdown'
)

# ─── Tabs content ───

In [146]:
tabs_1 = [dbc.Row([
            dbc.Col(dcc.Graph(id= 'total_matches')),
            dbc.Col(dcc.Graph(id= 'mean_kda')),
            dbc.Col(dcc.Graph(id= 'duration_mean'))]),
          dbc.Row([
            dbc.Col(dcc.Graph(id= 'top_15_wr'),
                              width= {'size': 6}),
            dbc.Col(dcc.Graph(id= 'top_15_count'),
                              width= {'size': 6})]),
          dbc.Row([
            dbc.Col(dcc.Graph(id= 'hist_duration'),
                              width= {'size': 6}),
            dbc.Col(dcc.Graph(id= 'hist_LP'),
                              width= {'size': 6})])
            ]
tabs_2 = [dbc.Row([
            dbc.Col(dcc.Graph(id= 'group_mean'))]),
          dbc.Row([
            dbc.Col(dcc.Graph(id= 'top_10_items'),
                              width= {'size': 6}),
            dbc.Col(dcc.Graph(id= 'top_5_role'),
                              width= {'size': 6})])
]
tabs_3 = [dbc.Row([
            dbc.Col(dcc.Graph(id= 'fig_kda_box')),
            dbc.Col(dcc.Graph(id= 'fig_corr'))]),
          dbc.Row([
            dbc.Col(dash_table.DataTable(id= 'table_data'))])
]

In [147]:

app.title = 'LoL Match Analytics'

app.layout = html.Div([
    dbc.Row(html.H1("Анализ матчей League of Legends")),
    dbc.Row([
        dbc.Col([html.Div('Фильтр регионов'),
                html.Div(region_selector)]),
        dbc.Col([html.Div('Фильтр роли'),
                html.Div(position_selector)]),
        dbc.Col([html.Div('Фильтр побед'),
                html.Div(win_selector)])
]),
    dbc.Tabs([
        dbc.Tab(tabs_1, label= ' Общий срез и KPI'),
        dbc.Tab(tabs_2, label= 'Герои и Роли'),
        dbc.Tab(tabs_3, label= 'Статистика')
    ]),
             ], style={'margin-left': '40px', 'margin-right': '40px', 'padding-bottom': '40px'}
)
# ─── Callback ────
@app.callback(
    [Output('total_matches', 'figure'),
     Output('hist_duration', 'figure'),
     Output('mean_kda', 'figure'),
     Output('duration_mean', 'figure'),
     Output('top_15_wr', 'figure'),
     Output('top_15_count', 'figure'),
     Output('group_mean', 'figure'),
     Output('hist_LP', 'figure'),
     Output('top_10_items', 'figure'),
     Output('top_5_role', 'figure'),
     Output('fig_kda_box', 'figure'),
     Output('fig_corr', 'figure'),
     Output('table_data', 'data')],
    [Input('region_selector', 'value'),
     Input('position_selector', 'value'),
     Input('win_selector', 'value')]
)

def update_graph(selected_regions, selected_positions, selected_win):
    chart_data = matches[matches['region'].isin(selected_regions)]
    chart_data = chart_data[chart_data['team_position'].isin(selected_positions)]
    if selected_win != 'all':
        chart_data = chart_data[chart_data['win_num'] == selected_win]
    chart_data_players = players[players['region_api'].isin(selected_regions)]
    
    champion_group = chart_data.groupby('champion')['win_num'].agg(['count', 'sum']).reset_index()

    #График 1
    total_matches = chart_data['match_id'].count()

    fig_total_matches = go.Figure(
        go.Indicator(value= total_matches,
                     number={'font': {'size': 48, 'color': '#60a5fa'}}))
    fig_total_matches.update_layout(title=f'Количество матчей',
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        height=150, margin=dict(l=20, r=20, t=30, b=20)
    )
    #График 2
    mean_kda = round((chart_data['kills'].mean() + chart_data['assists'].mean()) / chart_data['deaths'].mean(), 2)

    fig_mean_kda = go.Figure(
        go.Indicator(value= mean_kda,
                     number={'font': {'size': 48, 'color': '#fbbf24'}}))
    fig_mean_kda.update_layout(title=f'Среднее KDA',
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        height=150, margin=dict(l=20, r=20, t=30, b=20)
    )
    #График 3
    duration_mean = round(chart_data['duration_min'].mean())

    fig_duration_mean = go.Figure(
        go.Indicator(value= duration_mean,
                     number={'font': {'size': 48, 'color': '#4ade80'}, 'suffix': ' мин'}))
    fig_duration_mean.update_layout(title=f'Среднее время матча (мин)',
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        height=150, margin=dict(l=20, r=20, t=30, b=20)
    )
    # График 4
    count_df = champion_group[champion_group['count'] >= 1]
    top_count = count_df.nlargest(15, 'count').sort_values('count', ascending=True)

    fig_count = px.bar(top_count, x='count', y='champion', orientation='h',
                    title='Топ-15 чемпионов по популярности',
                    labels={'count': 'Количество', 'champion': 'Чемпион'},
                    text='count', color='count', color_continuous_scale='Viridis')
    fig_count.update_traces(texttemplate='%{text:.0f}', textposition='outside')
    fig_count.update_layout(
        xaxis_range=[0, top_count['count'].max() * 1.15],
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155')
    )
    # График 5
    winrate_df = champion_group.copy()
    winrate_df['win_rate'] = winrate_df['sum'] / winrate_df['count'] * 100
    winrate_df = winrate_df[winrate_df['count'] >= 1]
    top_wr = winrate_df.nlargest(15, 'win_rate').sort_values('win_rate', ascending=True)

    fig_wr = px.bar(top_wr, x='win_rate', y='champion', orientation='h',
                    title='Топ-15 чемпионов по винрейту',
                    labels={'win_rate': 'Винрейт (%)', 'champion': 'Чемпион'},
                    text='win_rate', color='win_rate', color_continuous_scale='Plasma')
    fig_wr.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig_wr.update_layout(
        xaxis_range=[0, 100], paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155')
    )
    # График 6
    hist_duration = px.histogram(chart_data, x='duration_min', nbins=30,
                            title='Длительность матчей',
                            labels={'duration_min': 'Длительность (мин)', 'count': 'Количество'},
                            color_discrete_sequence=['#636EFA'],
                            marginal='box')
    hist_duration.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155')
    )
    #График 7
    # Группировка по герою и позиции: средние kills, gold_earned и количество игр
    grouped_mean = chart_data.groupby(['champion', 'team_position'], as_index=False).agg(
        avg_kills=('kills', 'mean'),
        avg_gold=('gold_earned', 'mean'),
        count=('match_id', 'size')
    )
    fig_group_mean = px.scatter(grouped_mean, x='avg_kills', y='avg_gold', color='team_position',
                        size='count', hover_data=['champion'],
                        title='Связь убийств и золота по героям и ролям',
                        labels={'avg_kills': 'Средние убийства', 'avg_gold': 'Средний золотой доход'},
                        size_max=30, opacity=0.7,
                        color_discrete_map=POS_COLORS)
    fig_group_mean.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155')
    )
    # График 8: гистограмма распределения "LP"
    fig_hist = px.histogram(chart_data_players, x='league_points', nbins=30,
                            title='Распределение игроков по очкам лиги (LP)',
                            labels={'league_points': 'Количество LP', 'count': 'Количество'},
                            color_discrete_sequence=['#ab47bc'],
                            marginal='box')
    fig_hist.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155'),
        bargap=0.05
    )
    
    # График 9
    fig_top_10 = px.bar(top10, x='item_id', y='count', title='Топ-10 предметов у победивших',
                       color='count', color_continuous_scale='Teal')
    fig_top_10.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155')
    )

    # График 10
    fig_5_role = px.bar(role_top5, x='team_position', y='count', color='item_id',
                       title='Топ-5 предметов по ролям (победившие)',
                       color_discrete_sequence=px.colors.qualitative.Bold)
    fig_5_role.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155'), xaxis=dict(gridcolor='#334155')
    )

    # ── График 11: KDA box plot ────────────────────────────────────────
    box_data = []
    for pos in selected_positions:
        vals = chart_data[chart_data['team_position'] == pos]['kda'].clip(upper=15).values
        if len(vals) > 0:
            color = POS_COLORS.get(pos, '#888888')
            box_data.append(go.Box(
                y=vals,
                name=pos,
                marker_color=color,
                boxpoints=False,
                line=dict(width=1.5),
                fillcolor=color  # убрана прозрачность
            ))
    fig_kda_box = go.Figure(data=box_data)
    fig_kda_box.update_layout(
        title='Распределение KDA по ролям',
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        yaxis=dict(gridcolor='#334155', title='KDA', range=[0, 10]),
        xaxis=dict(gridcolor='#334155')
    )

    # ── График 12: correlation heatmap ─────────────────────────────────
    corr_cols = ['kills', 'deaths', 'assists', 'gold_earned', 'damage_to_champions', 
                 'minions_killed', 'vision_score', 'kda']
    corr = chart_data[corr_cols].corr().round(2)
    fig_corr = go.Figure(data=[go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.columns,
        colorscale='RdBu', zmid=0,
        text=corr.values, texttemplate='%{text}', textfont={'size': 10, 'color': '#e2e8f0'},
        hovertemplate='%{x} vs %{y}<br>r = %{z}<extra></extra>'
    )])
    fig_corr.update_layout(
        title='Корреляционная матрица',
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color='#cbd5e1', title_font_color='#e2e8f0',
        xaxis=dict(gridcolor='#334155', tickangle=-30),
        yaxis=dict(gridcolor='#334155')
    )

    # ── Таблица: champion stats ─────────────────────────────────────────
    table_df = chart_data.groupby('champion').agg(
        games=('match_id', 'count'),
        win_rate=('win_num', lambda x: round(x.mean() * 100, 1)),
        avg_kda=('kda', lambda x: round(x.mean(), 2)),
        avg_gold=('gold_earned', lambda x: round(x.mean())),
        avg_dmg=('damage_to_champions', lambda x: round(x.mean()))
    ).reset_index().sort_values('games', ascending=False).head(50)
    table_data = table_df.to_dict('records')
    
    return fig_total_matches, hist_duration, fig_mean_kda, fig_duration_mean, fig_wr, fig_count, fig_group_mean, fig_hist, fig_top_10, fig_5_role, fig_kda_box, fig_corr, table_data


if __name__ == '__main__':
    app.run(debug=True)